# Crop Guardian: paddy leaf diagnosis model

This notebook trains the on-device paddy (ধান) model and exports it in exactly the form the app reads.
The app's own code then decides "নিশ্চিত / নিশ্চিত নই / চিনি না" from this export, as it does for potato.

**What it does**

1. Pools three Bangladeshi field datasets, maps their folder names to one set of classes, and drops near-duplicate photos.
2. Splits by duplicate group (never the same photo, or a near copy, on both sides of a split): train / val / cal / test.
3. Fine-tunes EfficientNet-B0 with an extra **other** output, trained on diseases the app doesn't name.
4. Calibrates on the cal split: temperature, a conformal set at α = 0.1, and the "other" (out-of-distribution) threshold.
5. Tests on the held-out test split, on a disease it never saw (**narrow brown spot**), and, optionally, leave-one-dataset-out.
6. Exports `crop_guardian_paddy_v1.onnx`, its config, 12 parity images and a model-card section, zipped in the repo's own folder layout.

**Before you run it**

* Runtime → Change runtime type → **T4 GPU**.
* Download these datasets (each is a zip) and put the zips, unopened, in one Google Drive folder (default `MyDrive/crop-guardian/paddy-data/`):
  * RiceLeafBD (Sylhet & Dhaka, Aman): https://data.mendeley.com/datasets/kx9rx8p2mz
  * RiceLeafDisease-BD5 (Savar, Dhaka, Oct–Nov 2024): https://data.mendeley.com/datasets/sjkptpgv79
  * Dhan-Shomadhan (field and white background): https://data.mendeley.com/datasets/znsxdctwtt
* Check each dataset's licence on its page before publishing results.

**After it finishes**: download `paddy_export.zip`, unzip it at the root of the Crop-Guardian repo, add `'paddy'` to `PHOTO_CROPS` in `src/lib/diagnosis/crops.ts`, and run `npm test`.

In [ ]:
# Settings. SMOKE builds a tiny synthetic dataset and runs every step in a few minutes on a CPU,
# to check the notebook itself; leave it False for the real model.
SMOKE = False

DRIVE_FOLDER = 'crop-guardian/paddy-data'   # under MyDrive: the dataset zips
WORK = '/content/paddy'                      # unzipped data and outputs (not on Drive)

SEED = 42
EPOCHS = 20
PATIENCE = 5            # stop when val macro-F1 hasn't improved for this many epochs
BATCH = 32
LR = 5e-4
IMG_RESIZE = 256        # shorter side, then...
IMG_CROP = 224          # ...centre crop: the app does exactly this (config.preprocess)
ALPHA = 0.1             # conformal: the set holds the true disease ≥ 90% of the time
OOD_FPR = 0.05          # share of known-disease cal photos allowed to come out "unknown"
DUP_HAMMING = 4         # pHash distance at or below which two photos count as the same
INCLUDE_WHITE_BACKGROUND = False   # Dhan-Shomadhan's white-background photos are not what a farmer's phone sees
RUN_LODO = True         # leave-one-dataset-out runs (about 3 extra trainings)
LODO_EPOCHS = 12
# Folders of other leaves (e.g. the potato dataset) to add to "other", so non-paddy leaves come out "unknown".
OTHER_EXTRA_DIRS = []

In [ ]:
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnx', 'onnxruntime', 'onnxscript', 'imagehash'], check=True)

import json, math, random, re, shutil, time, zipfile, hashlib
from collections import Counter, defaultdict
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps
import imagehash
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, roc_auc_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
WORK = Path('/tmp/paddy-smoke' if SMOKE else WORK)
DATA = WORK / 'data'
OUT = WORK / 'export'
if SMOKE:
    EPOCHS, PATIENCE, LODO_EPOCHS, BATCH = 2, 2, 1, 16
print('device:', DEVICE, '| torch', torch.__version__)

## 1. Classes

The six classes and their Bengali names must match `labelsBn` for paddy in `src/lib/diagnosis/crops.ts` (a test checks).
**other** is the extra output the app uses only to say "চিনি না". **narrow brown spot** is never trained on: it is the unseen-disease test.

In [ ]:
CLASSES = ['bacterial_leaf_blight', 'blast', 'brown_spot', 'healthy', 'sheath_blight', 'tungro']
OTHER = 'other'
ALL_OUTPUTS = CLASSES + [OTHER]
LABELS_BN = {
    'bacterial_leaf_blight': 'পাতা পোড়া',
    'blast': 'ব্লাস্ট',
    'brown_spot': 'বাদামি দাগ',
    'healthy': 'সুস্থ',
    'sheath_blight': 'খোল পোড়া',
    'tungro': 'টুংরো',
    'other': 'অন্য রোগ',
}
OTHER_TRAIN = ['leaf_scald']          # trained as "other"
UNSEEN = ['narrow_brown_spot']        # never trained: tests "চিনি না"

# Folder (or CSV label) names in the datasets → class. Names are compared lower-case with letters only,
# and must match whole, so "narrow brown spot" is never read as "brown spot".
SYNONYMS = {
    'bacterialleafblight': 'bacterial_leaf_blight', 'bacterialblight': 'bacterial_leaf_blight', 'blb': 'bacterial_leaf_blight',
    'blast': 'blast', 'riceblast': 'blast', 'leafblast': 'blast',
    'brownspot': 'brown_spot', 'brownleafspot': 'brown_spot', 'ricebrownspot': 'brown_spot',
    'healthy': 'healthy', 'healthyleaf': 'healthy', 'normal': 'healthy', 'normalleaf': 'healthy', 'healthyrice': 'healthy',
    'sheathblight': 'sheath_blight', 'steathblight': 'sheath_blight', 'ricesheathblight': 'sheath_blight',
    'tungro': 'tungro', 'ricetungro': 'tungro', 'tungrovirus': 'tungro', 'riceturngo': 'tungro', 'turngo': 'tungro',
    'leafscald': 'leaf_scald', 'leafscaled': 'leaf_scald', 'riceleafscald': 'leaf_scald',
    'narrowbrownspot': 'narrow_brown_spot', 'narrowbrownleafspot': 'narrow_brown_spot',
}
norm = lambda s: re.sub(r'[^a-z]', '', str(s).lower())
IMAGE_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

## 2. Data

In [ ]:
def make_smoke_data(root: Path):
    """Three fake 'datasets' of coloured shapes, laid out like the real ones, to exercise every step."""
    from PIL import ImageDraw
    rng = np.random.default_rng(SEED)
    layout = {
        'RiceLeafBD': ['Healthy', 'Bacterial Leaf Blight', 'Brown Spot', 'Tungro Virus'],
        'RiceLeafDisease-BD5': ['Blast', 'Narrow Brown Spot', 'Sheath Blight', 'Tungro', 'Normal Leaf'],
        'Dhan-Shomadhan': ['Field Background/Brown Spot', 'Field Background/Leaf Scaled', 'Field Background/Rice Blast',
                           'Field Background/Rice Turngo', 'Field Background/Steath Blight', 'White Background/Rice Blast'],
    }
    colour = {'healthy': (60, 140, 60), 'bacterial_leaf_blight': (220, 210, 140), 'brown_spot': (120, 70, 30), 'tungro': (230, 160, 50),
              'blast': (190, 190, 180), 'narrow_brown_spot': (140, 80, 40), 'sheath_blight': (150, 170, 140), 'leaf_scald': (200, 120, 90)}
    for ds, folders in layout.items():
        for folder in folders:
            cls = SYNONYMS[norm(folder.split('/')[-1])]
            d = root / ds / folder
            d.mkdir(parents=True, exist_ok=True)
            for i in range(24):
                img = Image.new('RGB', (320, 240), tuple(int(c) for c in rng.integers(30, 90, 3)))
                draw = ImageDraw.Draw(img)
                for _ in range(6):
                    x, y = rng.integers(20, 280), rng.integers(20, 200)
                    c = tuple(int(v) for v in np.clip(np.array(colour[cls]) + rng.integers(-25, 25, 3), 0, 255))
                    draw.ellipse([x, y, x + rng.integers(15, 40), y + rng.integers(8, 25)], fill=c)
                img.save(d / f'{i}.jpg', quality=90)
        # One exact duplicate across datasets, to check de-duplication.
    shutil.copy(root / 'RiceLeafBD' / 'Healthy' / '0.jpg', root / 'RiceLeafDisease-BD5' / 'Normal Leaf' / 'dup.jpg')

if DATA.exists():
    shutil.rmtree(DATA)
DATA.mkdir(parents=True)
if SMOKE:
    make_smoke_data(DATA)
else:
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
    src = Path('/content/drive/MyDrive') / DRIVE_FOLDER
    zips = sorted(src.glob('*.zip'))
    assert zips, f'No dataset zips in {src}'
    for z in zips:
        target = DATA / z.stem
        with zipfile.ZipFile(z) as f:
            f.extractall(target)
        # Some datasets zip further zips inside.
        for inner in list(target.rglob('*.zip')):
            with zipfile.ZipFile(inner) as f:
                f.extractall(inner.with_suffix(''))
            inner.unlink()
        print('unzipped', z.name)

In [ ]:
def class_from_path(p: Path, root: Path):
    for part in reversed(p.relative_to(root).parts[:-1]):
        if norm(part) in SYNONYMS:
            return SYNONYMS[norm(part)]
    return None

def labels_from_csvs(ds_root: Path):
    """Image name → class from any CSV in the dataset that has a column of image names and a column of class names."""
    found = {}
    for csv in ds_root.rglob('*.csv'):
        try:
            df = pd.read_csv(csv)
        except Exception:
            continue
        name_col = next((c for c in df.columns if df[c].astype(str).str.lower().str.contains(r'\.(?:jpe?g|png)$').mean() > 0.5), None)
        label_col = next((c for c in df.columns if df[c].map(norm).isin(SYNONYMS).mean() > 0.5), None)
        if name_col and label_col:
            for n, l in zip(df[name_col].astype(str), df[label_col]):
                if norm(l) in SYNONYMS:
                    found[Path(n).name] = SYNONYMS[norm(l)]
            print(f'  labels from {csv.relative_to(ds_root)}: {len(found)}')
    return found

rows, unmapped = [], Counter()
for ds_root in sorted(p for p in DATA.iterdir() if p.is_dir()):
    csv_labels = labels_from_csvs(ds_root)
    for p in ds_root.rglob('*'):
        if p.suffix.lower() not in IMAGE_EXT or p.name.startswith('.'):
            continue
        cls = class_from_path(p, ds_root) or csv_labels.get(p.name)
        if cls is None:
            unmapped[str(p.parent.relative_to(DATA))] += 1
            continue
        white = any('white' in norm(part) for part in p.relative_to(ds_root).parts)
        rows.append({'path': str(p), 'dataset': ds_root.name, 'cls': cls, 'white': white})
for extra in OTHER_EXTRA_DIRS:
    for p in Path(extra).rglob('*'):
        if p.suffix.lower() in IMAGE_EXT:
            rows.append({'path': str(p), 'dataset': 'extra_other', 'cls': 'extra_other', 'white': False})

df = pd.DataFrame(rows)
if not INCLUDE_WHITE_BACKGROUND:
    print('dropping white-background photos:', int(df.white.sum()))
    df = df[~df.white].reset_index(drop=True)
if unmapped:
    print('Folders with images but no class (add them to SYNONYMS if they are a class):')
    for k, v in unmapped.most_common():
        print(f'  {v:5d}  {k}')
print(pd.crosstab(df.cls, df.dataset, margins=True))
missing = [c for c in CLASSES if c not in set(df.cls)]
assert not missing, f'No photos for {missing}: check the folder names above'

## 3. Near-duplicates

The same leaf photographed twice, or a photo resized into two datasets, would leak between train and test and flatter the score.
Photos within `DUP_HAMMING` bits of perceptual hash form one group; a group stays on one side of every split.
A group whose photos carry different labels is dropped as contradictory.

In [ ]:
def phash(path):
    try:
        with Image.open(path) as im:
            return int(str(imagehash.phash(ImageOps.exif_transpose(im).convert('RGB'))), 16)
    except Exception as e:
        print('unreadable, dropped:', path, e)
        return None

t0 = time.time()
df['hash'] = [phash(p) for p in df.path]
df = df[df.hash.notna()].reset_index(drop=True)
hashes = np.array([np.uint64(h) for h in df.hash], dtype=np.uint64)
bits = np.unpackbits(hashes.view(np.uint8).reshape(-1, 8), axis=1)   # n × 64
parent = list(range(len(df)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
for i in range(len(df)):
    dist = (bits[i + 1:] != bits[i]).sum(1)
    for j in np.nonzero(dist <= DUP_HAMMING)[0] + i + 1:
        a, b = find(i), find(int(j))
        if a != b:
            parent[b] = a
df['group'] = [find(i) for i in range(len(df))]
labels_per_group = df.groupby('group').cls.nunique()
conflicted = set(labels_per_group[labels_per_group > 1].index)
print(f'hashing took {time.time() - t0:.0f}s; {len(df)} photos in {df.group.nunique()} groups; '
      f'{(df.groupby("group").size() > 1).sum()} groups with near-copies; {len(conflicted)} contradictory groups dropped')
df = df[~df.group.isin(conflicted)].reset_index(drop=True)

## 4. Splits

Per class, groups are shuffled and cut 70 / 10 / 10 / 10 into train, val (early stopping), cal (temperature, conformal and OOD thresholds) and test.
Unseen-disease photos go only to the OOD test.

In [ ]:
df['target'] = df.cls.map(lambda c: c if c in CLASSES else (OTHER if (c in OTHER_TRAIN or c == 'extra_other') else None))
df['split'] = None
df.loc[df.cls.isin(UNSEEN), 'split'] = 'unseen'
rng = random.Random(SEED)
for cls, part in df[df.split.isna() & df.target.notna()].groupby('cls'):
    groups = sorted(part.group.unique())
    rng.shuffle(groups)
    n = len(groups)
    cuts = [int(n * 0.7), int(n * 0.8), int(n * 0.9)]
    for name, gs in zip(['train', 'val', 'cal', 'test'], np.split(np.array(groups), cuts)):
        df.loc[df.group.isin(gs) & (df.cls == cls), 'split'] = name
df = df[df.split.notna()].reset_index(drop=True)
print(pd.crosstab(df.cls, df.split, margins=True))

## 5. Training

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
eval_tf = transforms.Compose([
    transforms.Resize(IMG_RESIZE, interpolation=transforms.InterpolationMode.BILINEAR, antialias=True),
    transforms.CenterCrop(IMG_CROP),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_CROP, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.03),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
TARGET_INDEX = {c: i for i, c in enumerate(ALL_OUTPUTS)}

def load_rgb(path):
    with Image.open(path) as im:
        return ImageOps.exif_transpose(im).convert('RGB')

class Photos(Dataset):
    def __init__(self, frame, tf):
        self.paths = list(frame.path)
        self.y = [TARGET_INDEX.get(t, -1) for t in frame.target.fillna('')]
        self.tf = tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.tf(load_rgb(self.paths[i])), self.y[i]

WORKERS = 2 if IN_COLAB else 0
def loader(frame, train=False):
    ds = Photos(frame, train_tf if train else eval_tf)
    if not train:
        return DataLoader(ds, batch_size=BATCH * 2, shuffle=False, num_workers=WORKERS)
    # Every class (and "other") equally often per epoch.
    counts = Counter(ds.y)
    weights = [1.0 / counts[y] for y in ds.y]
    sampler = WeightedRandomSampler(weights, num_samples=len(ds), replacement=True, generator=torch.Generator().manual_seed(SEED))
    return DataLoader(ds, batch_size=BATCH, sampler=sampler, num_workers=WORKERS, drop_last=len(ds) > BATCH)

def new_model():
    weights = None if SMOKE else models.EfficientNet_B0_Weights.IMAGENET1K_V1
    m = models.efficientnet_b0(weights=weights)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(ALL_OUTPUTS))
    return m.to(DEVICE)

@torch.no_grad()
def logits_of(model, frame):
    model.eval()
    out = [model(x.to(DEVICE)).float().cpu() for x, _ in loader(frame)]
    return torch.cat(out).numpy() if out else np.zeros((0, len(ALL_OUTPUTS)), dtype=np.float32)

def macro_f1(model, frame):
    known = frame[frame.target.isin(CLASSES)]
    pred = logits_of(model, known)[:, :len(CLASSES)].argmax(1)
    return f1_score(known.target.map(TARGET_INDEX), pred, average='macro')

def train(frame_train, frame_val, epochs):
    model = new_model()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    steps = epochs * max(1, len(loader(frame_train, train=True)))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=DEVICE == 'cuda')
    best, best_state, stale = -1.0, None, 0
    for epoch in range(epochs):
        model.train()
        t0, total = time.time(), 0.0
        for x, y in loader(frame_train, train=True):
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.autocast(device_type='cuda', enabled=DEVICE == 'cuda'):
                loss = F.cross_entropy(model(x), y, label_smoothing=0.05)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            total += loss.item()
        f1 = macro_f1(model, frame_val)
        print(f'epoch {epoch + 1:2d}  loss {total:8.2f}  val macro-F1 {f1:.4f}  ({time.time() - t0:.0f}s)')
        if f1 > best:
            best, best_state, stale = f1, {k: v.detach().clone() for k, v in model.state_dict().items()}, 0
        else:
            stale += 1
            if stale >= PATIENCE:
                print('early stop')
                break
    model.load_state_dict(best_state)
    return model, best

split = {s: df[df.split == s] for s in ['train', 'val', 'cal', 'test', 'unseen']}
model, best_val = train(split['train'], split['val'], EPOCHS)
print('best val macro-F1:', round(best_val, 4))

## 6. Calibration (cal split only)

The same three numbers the potato config has, fitted the same way the app uses them (`src/lib/diagnosis/decide.ts`):

* **temperature** T on the six disease logits, by minimising cal NLL;
* **conformal q̂**: with p = softmax(disease logits / T), a disease is in the set when p ≥ 1 − q̂; q̂ is the ⌈(n+1)(1−α)⌉/n quantile of 1 − p(true disease) on cal;
* **OOD threshold** on p_other = softmax(all 7 logits)[other]: the (1 − OOD_FPR) quantile over the known-disease cal photos.

In [ ]:
K = len(CLASSES)
def softmax(z, axis=1):
    z = z - z.max(axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis, keepdims=True)

cal = split['cal']
cal_logits = logits_of(model, cal)
cal_known = cal.target.isin(CLASSES).to_numpy()
cal_y = cal.target[cal_known].map(TARGET_INDEX).to_numpy()
cal_dl = cal_logits[cal_known, :K]

log_t = torch.zeros(1, requires_grad=True)
lbfgs = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200)
zt, yt = torch.tensor(cal_dl), torch.tensor(cal_y)
def closure():
    lbfgs.zero_grad()
    loss = F.cross_entropy(zt / log_t.exp(), yt)
    loss.backward()
    return loss
lbfgs.step(closure)
TEMPERATURE = float(log_t.detach().exp())

p_cal = softmax(cal_dl / TEMPERATURE)
scores = 1 - p_cal[np.arange(len(cal_y)), cal_y]
n = len(scores)
QHAT = float(np.quantile(scores, min(1.0, math.ceil((n + 1) * (1 - ALPHA)) / n), method='higher'))

p_other_cal = softmax(cal_logits)[:, K]
OOD_THRESHOLD = float(np.quantile(p_other_cal[cal_known], 1 - OOD_FPR, method='higher'))
print(f'T = {TEMPERATURE:.4f}   q̂ = {QHAT:.4f} (a disease is in the set at p ≥ {1 - QHAT:.4f})   OOD threshold = {OOD_THRESHOLD:.6f}')

def decide(logits):
    """The app's decision (decide.ts), for one photo's 7 logits."""
    p = softmax(logits[None, :K] / TEMPERATURE)[0]
    ood = softmax(logits[None])[0, K]
    cands = [CLASSES[i] for i in np.argsort(-p) if p[i] >= 1 - QHAT]
    if ood > OOD_THRESHOLD or not cands:
        return 'unknown', [], p, ood
    return ('confident' if len(cands) == 1 else 'uncertain'), cands, p, ood

## 7. Test results

Everything below is on photos the model and the calibration never saw.

In [ ]:
def ece(probs, y, bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    edges = np.linspace(0, 1, bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(total)

def app_outcome(truth, decision, cands):
    """What the farmer is told, judged like scripts/eval-field-photos.ts."""
    if truth not in CLASSES:
        return 'right' if decision == 'unknown' else 'misleading'
    if decision == 'unknown':
        return 'unknown'
    if truth not in cands:
        return 'misleading'
    return 'right' if decision == 'confident' else 'shortlist'

test = split['test']
test_logits = logits_of(model, test)
known = test.target.isin(CLASSES).to_numpy()
y = test.target[known].map(TARGET_INDEX).to_numpy()
dl = test_logits[known, :K]
pred = dl.argmax(1)

metrics = {
    'n': {s: int((df.split == s).sum()) for s in ['train', 'val', 'cal', 'test', 'unseen']},
    'n_by_class_test': test.cls.value_counts().to_dict(),
    'accuracy': float(accuracy_score(y, pred)),
    'macro_f1': float(f1_score(y, pred, average='macro')),
    'ece_before': ece(softmax(dl), y),
    'ece_after': ece(softmax(dl / TEMPERATURE), y),
    'temperature': TEMPERATURE, 'qhat': QHAT, 'ood_threshold': OOD_THRESHOLD, 'alpha': ALPHA,
}
report = classification_report(y, pred, labels=list(range(K)), target_names=CLASSES, digits=3, output_dict=True, zero_division=0)
metrics['per_class'] = {c: {k: round(report[c][k], 3) for k in ['precision', 'recall', 'f1-score', 'support']} for c in CLASSES}
cm = confusion_matrix(y, pred, labels=list(range(K)))
metrics['confusion'] = cm.tolist()

decisions = [decide(l) for l in test_logits]
sets_known = [d for d, k in zip(decisions, known) if k]
metrics['coverage'] = float(np.mean([CLASSES[t] in d[1] for d, t in zip(sets_known, y) if d[0] != 'unknown'] or [0]))
metrics['coverage_counting_unknown_as_miss'] = float(np.mean([CLASSES[t] in d[1] for d, t in zip(sets_known, y)]))
metrics['avg_set_size'] = float(np.mean([len(d[1]) for d in sets_known if d[0] != 'unknown'] or [0]))
metrics['decisions_known'] = dict(Counter(d[0] for d in sets_known))
outcomes = pd.DataFrame({'cls': test.cls.values, 'outcome': [app_outcome(t, d[0], d[1]) for t, d in zip(test.cls, decisions)]})
metrics['app_outcomes'] = pd.crosstab(outcomes.cls, outcomes.outcome).to_dict(orient='index')
test_ds = test[known].dataset.to_numpy()
metrics['by_dataset'] = {ds: float(accuracy_score(y[test_ds == ds], pred[test_ds == ds])) for ds in sorted(set(test_ds))}

# Out of distribution: known test photos vs a disease never trained on, and vs held-out "other" photos.
p_other_test = softmax(test_logits)[:, K]
unseen = split['unseen']
ood = {'false_unknown_on_known': float(np.mean([d[0] == 'unknown' for d in sets_known]))}
if len(unseen):
    unseen_logits = logits_of(model, unseen)
    p_other_unseen = softmax(unseen_logits)[:, K]
    unseen_dec = [decide(l) for l in unseen_logits]
    ood['unseen_n'] = len(unseen)
    ood['auroc_known_vs_unseen'] = float(roc_auc_score(np.r_[np.zeros(known.sum()), np.ones(len(unseen))], np.r_[p_other_test[known], p_other_unseen]))
    ood['unseen_called_unknown'] = float(np.mean([d[0] == 'unknown' for d in unseen_dec]))
    ood['unseen_called_confident'] = dict(Counter(d[1][0] for d in unseen_dec if d[0] == 'confident'))
other_test = ~known
if other_test.any():
    ood['other_n'] = int(other_test.sum())
    ood['auroc_known_vs_other'] = float(roc_auc_score(np.r_[np.zeros(known.sum()), np.ones(other_test.sum())], np.r_[p_other_test[known], p_other_test[other_test]]))
    ood['other_called_unknown'] = float(np.mean([decisions[i][0] == 'unknown' for i in np.nonzero(other_test)[0]]))
metrics['ood'] = ood

print(json.dumps({k: v for k, v in metrics.items() if k not in ('confusion', 'per_class', 'app_outcomes')}, indent=1, ensure_ascii=False))
print(pd.DataFrame(cm, index=CLASSES, columns=CLASSES))
print(pd.DataFrame(metrics['app_outcomes']).T.fillna(0).astype(int))

## 8. Leave one dataset out

The honest question for a farmer's phone: does the model work on photos from fields it has never seen?
For each dataset, train on the others and test on all of its photos, for the classes the others cover.
(Calibration here comes from the others' cal split, so these numbers show the drift a new district would see.)

In [ ]:
lodo = {}
if RUN_LODO:
    for ds in sorted(set(df.dataset) - {'extra_other'}):
        rest = df[df.dataset != ds]
        covered = set(rest[rest.split == 'train'].target) & set(CLASSES)
        held = df[(df.dataset == ds) & df.target.isin(covered)]
        if held.empty or len(covered) < 2:
            continue
        print(f'--- without {ds}: testing on {len(held)} photos of {sorted(set(held.target))}')
        m, _ = train(rest[rest.split == 'train'], rest[rest.split == 'val'], LODO_EPOCHS)
        hl = logits_of(m, held)
        hy = held.target.map(TARGET_INDEX).to_numpy()
        hp = hl[:, :K].argmax(1)
        dec = [decide(l) for l in hl]
        lodo[ds] = {
            'n': len(held), 'classes': sorted(set(held.target)),
            'accuracy': float(accuracy_score(hy, hp)), 'macro_f1': float(f1_score(hy, hp, average='macro')),
            'misleading': float(np.mean([app_outcome(CLASSES[t], d[0], d[1]) == 'misleading' for t, d in zip(hy, dec)])),
            'unknown': float(np.mean([d[0] == 'unknown' for d in dec])),
        }
        print(lodo[ds])
        del m
metrics['leave_one_dataset_out'] = lodo

## 9. Export

The ONNX model, the config in the app's format, and a check that ONNX Runtime gives the same logits and decisions as PyTorch.

In [ ]:
if OUT.exists():
    shutil.rmtree(OUT)
MODEL_DIR = OUT / 'public' / 'model' / 'paddy'
PARITY_DIR = OUT / 'src' / 'lib' / 'diagnosis' / '__fixtures__' / 'parity-paddy'
DOCS_DIR = OUT / 'docs'
for d in (MODEL_DIR, PARITY_DIR, DOCS_DIR):
    d.mkdir(parents=True)
MODEL_FILE = 'crop_guardian_paddy_v1.onnx'
CONFIG_FILE = 'crop_guardian_paddy_config.json'

cpu_model = model.to('cpu').eval()
dummy = torch.randn(1, 3, IMG_CROP, IMG_CROP)
onnx_path = MODEL_DIR / MODEL_FILE
export_kwargs = dict(input_names=['image'], output_names=['logits'], dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}}, opset_version=17)
try:
    torch.onnx.export(cpu_model, dummy, str(onnx_path), dynamo=False, **export_kwargs)
except TypeError:   # older torch without the dynamo switch
    torch.onnx.export(cpu_model, dummy, str(onnx_path), **export_kwargs)
assert not list(MODEL_DIR.glob('*.data')), 'the model must be one file (no external weights)'
model.to(DEVICE)

import onnx, onnxruntime as ort
onnx.checker.check_model(str(onnx_path))
session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
check = split['test'].head(64)
torch_logits = logits_of(model, check)
onnx_logits = np.concatenate([session.run(['logits'], {'image': x.numpy()})[0] for x, _ in loader(check)])
diff = float(np.abs(torch_logits - onnx_logits).max())
same = all(decide(a)[:2] == decide(b)[:2] for a, b in zip(torch_logits, onnx_logits))
print(f'ONNX vs PyTorch: max logit difference {diff:.2e}; same decisions: {same}; model {onnx_path.stat().st_size / 1e6:.1f} MB')
assert diff < 1e-2 and same

today = date.today().isoformat()
config = {
    'model_file': MODEL_FILE,
    'input_name': 'image',
    'output_name': 'logits',
    'classes': ALL_OUTPUTS,
    'labels_bn': LABELS_BN,
    'preprocess': {
        'resize_shorter_side': IMG_RESIZE, 'center_crop': IMG_CROP, 'scale': 1 / 255,
        'mean': MEAN, 'std': STD, 'layout': 'NCHW', 'dtype': 'float32',
    },
    'temperature': TEMPERATURE,
    'conformal_qhat': QHAT,
    'conformal_alpha': ALPHA,
    'ood': {'method': 'p_other', 'threshold': OOD_THRESHOLD},
    'decision': [
        f'disease_logits = logits[0:{K}]',
        'p = softmax(disease_logits / temperature)',
        f"ood_score = softmax(all logits)[{K}] if method == 'p_other'",
        f'set = [i for i in 0..{K - 1} if p[i] >= 1 - conformal_qhat]',
        "if ood_score > threshold or set is empty -> 'unknown' (চিনি না)",
        "elif len(set) == 1 -> 'confident' (নিশ্চিত), label = classes[set[0]]",
        "else -> 'uncertain' (নিশ্চিত নই), candidates = set sorted by p desc",
    ],
    'provenance': (f'paddy_training.ipynb, {today}: EfficientNet-B0 + "other" ({", ".join(OTHER_TRAIN)}); '
                   f'datasets {", ".join(sorted(set(df.dataset)))}; group-aware split 70/10/10/10; '
                   f'thresholds fitted on the Bangladeshi field cal split; unseen test: {", ".join(UNSEEN)}'),
}
(MODEL_DIR / CONFIG_FILE).write_text(json.dumps(config, ensure_ascii=False, indent=2) + '\n')
print('wrote', MODEL_DIR / CONFIG_FILE)

## 10. Parity images

Twelve test photos (4 healthy, 4 diseased, 4 of the unseen disease) with the notebook's logits and decisions.
`parity.test.ts` runs the exported model on them through the app's own preprocessing and must reach the same decisions.
Photos close to a threshold are skipped, since the app's resizing differs slightly from PIL's.

In [ ]:
def margin(logits):
    """How far a photo is from changing its decision or its candidates' order."""
    d, c, p, o = decide(logits)
    gaps = [abs(o - OOD_THRESHOLD) / max(OOD_THRESHOLD, 1e-6), np.min(np.abs(p - (1 - QHAT)))]
    if len(c) > 1:
        ps = np.sort(p)[::-1][:len(c)]
        gaps.append(np.min(-np.diff(ps)))
    return min(gaps)

def pick(frame, decision, k):
    """The k clearest photos with this decision, else the clearest others."""
    if frame.empty:
        return frame
    lg = onnx_session_logits(frame)
    order = sorted(range(len(frame)), key=lambda i: -margin(lg[i]))
    chosen = [i for i in order if decide(lg[i])[0] == decision][:k]
    if len(chosen) < k:
        print(f'only {len(chosen)} "{decision}" photos among {sorted(set(frame.cls))}; adding the clearest others')
        chosen += [i for i in order if i not in chosen][:k - len(chosen)]
    return frame.iloc[chosen]

def onnx_session_logits(frame):
    return np.concatenate([session.run(['logits'], {'image': x.numpy()})[0] for x, _ in loader(frame)]) if len(frame) else np.zeros((0, K + 1))

picks = [('healthy', pick(test[test.cls == 'healthy'], 'confident', 4))]
sick = test[test.target.isin(CLASSES) & (test.cls != 'healthy')]
picks.append(('disease', pick(sick.groupby('cls').head(3), 'confident', 4)))
if len(unseen):
    picks.append(('unseen_disease', pick(unseen, 'unknown', 4)))

expected = []
for group, frame in picks:
    for i, row in enumerate(frame.itertuples()):
        name = f'{group}_{i}.jpg'
        img = load_rgb(row.path)
        img.thumbnail((512, 512))
        img.save(PARITY_DIR / name, quality=92)
        # Logits of the saved file, so the app and the notebook look at the same pixels.
        x = eval_tf(load_rgb(PARITY_DIR / name))[None].numpy()
        lg = session.run(['logits'], {'image': x})[0][0]
        d, cands, p, o = decide(lg)
        expected.append({
            'file': name, 'group': row.cls, 'source': str(Path(row.path).relative_to(DATA)),
            'logits': [round(float(v), 5) for v in lg], 'decision': d, 'candidates': cands,
            'probs': [round(float(v), 4) for v in p], 'ood_score': round(float(o), 4),
        })
(PARITY_DIR / 'expected.json').write_text(json.dumps(expected, ensure_ascii=False, indent=2) + '\n')
assert expected, 'no parity images'
print(pd.DataFrame(expected)[['file', 'group', 'decision', 'candidates', 'ood_score']])

## 11. Model card numbers and the zip

In [ ]:
pct = lambda v: f'{100 * v:.1f}%'
lines = [
    '## Paddy model (v1)',
    '',
    f'Generated by `notebooks/paddy_training.ipynb` on {today}. Every number is from photos the model and its calibration never saw.',
    '',
    '| | |', '|---|---|',
    '| Task | Classify one paddy leaf (or sheath) photo as ' + ', '.join(f'{LABELS_BN[c]} ({c})' for c in CLASSES) + ', or say "unknown" |',
    f'| Architecture | EfficientNet-B0 (ImageNet), fine-tuned, + an "other" output; `{MODEL_FILE}`, {onnx_path.stat().st_size / 1e6:.1f} MB |',
    f'| Data | {", ".join(sorted(set(df.dataset)))}: Bangladeshi field photos{"" if INCLUDE_WHITE_BACKGROUND else " (white-background photos left out)"}; near-duplicates grouped (pHash ≤ {DUP_HAMMING}) and kept on one side of every split |',
    f'| Split (photos) | train {metrics["n"]["train"]}, val {metrics["n"]["val"]}, cal {metrics["n"]["cal"]}, test {metrics["n"]["test"]}, unseen {metrics["n"]["unseen"]} |',
    f'| "Other" (trained) | {", ".join(OTHER_TRAIN)}{" + " + str(len(OTHER_EXTRA_DIRS)) + " extra folder(s)" if OTHER_EXTRA_DIRS else ""} |',
    f'| Unseen disease (never trained) | {", ".join(UNSEEN)} |',
    '',
    '### Test split',
    '',
    f'* Accuracy {pct(metrics["accuracy"])}, macro-F1 {metrics["macro_f1"]:.3f} (top-1 over the six classes)',
    f'* Calibration: ECE {metrics["ece_before"]:.3f} before, {metrics["ece_after"]:.3f} after temperature scaling (T = {TEMPERATURE:.3f})',
    f'* Conformal set at α = {ALPHA}: holds the true class for {pct(metrics["coverage"])} of answered photos (target ≥ {pct(1 - ALPHA)}), average size {metrics["avg_set_size"]:.2f}',
    f'* Decisions on known diseases: ' + ', '.join(f'{k} {v}' for k, v in sorted(metrics["decisions_known"].items())),
    f'* Known photos wrongly called "unknown": {pct(ood["false_unknown_on_known"])}',
]
if 'auroc_known_vs_unseen' in ood:
    lines.append(f'* Unseen disease ({", ".join(UNSEEN)}, {ood["unseen_n"]} photos): AUROC of p_other {ood["auroc_known_vs_unseen"]:.3f}; called "unknown" {pct(ood["unseen_called_unknown"])}; confidently named as ' + (', '.join(f'{k} {v}' for k, v in ood['unseen_called_confident'].items()) or 'nothing'))
if 'auroc_known_vs_other' in ood:
    lines.append(f'* Held-out "other" photos ({ood["other_n"]}): AUROC {ood["auroc_known_vs_other"]:.3f}; called "unknown" {pct(ood["other_called_unknown"])}')
lines += ['', '| Class | Precision | Recall | F1 | Photos |', '|---|--:|--:|--:|--:|']
lines += [f'| {LABELS_BN[c]} ({c}) | {r["precision"]:.3f} | {r["recall"]:.3f} | {r["f1-score"]:.3f} | {int(r["support"])} |' for c, r in metrics['per_class'].items()]
lines += ['', 'Confusion matrix (rows: truth, columns: top-1):', '', '| | ' + ' | '.join(CLASSES) + ' |', '|---|' + '--:|' * K]
lines += [f'| {c} | ' + ' | '.join(str(v) for v in row) + ' |' for c, row in zip(CLASSES, cm)]
lines += ['', 'Per dataset (test accuracy): ' + ', '.join(f'{k} {pct(v)}' for k, v in metrics['by_dataset'].items())]
if lodo:
    lines += ['', '### Leave one dataset out', '', '| Held-out dataset | Photos | Classes | Accuracy | Macro-F1 | Misleading | "Unknown" |', '|---|--:|---|--:|--:|--:|--:|']
    lines += [f'| {k} | {v["n"]} | {", ".join(v["classes"])} | {pct(v["accuracy"])} | {v["macro_f1"]:.3f} | {pct(v["misleading"])} | {pct(v["unknown"])} |' for k, v in lodo.items()]
lines += ['', '"Misleading" is a confident wrong answer, or a shortlist without the true disease.', '']
card = '\n'.join(lines)
(DOCS_DIR / 'model-card-paddy.md').write_text(card)
(DOCS_DIR / 'paddy-metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=1, default=str))
print(card)

zip_path = WORK / 'paddy_export.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file():
            z.write(p, p.relative_to(OUT))
print('\n', zip_path, f'{zip_path.stat().st_size / 1e6:.1f} MB')
if IN_COLAB and not SMOKE:
    from google.colab import files
    files.download(str(zip_path))

## Next

1. Unzip `paddy_export.zip` at the root of the Crop-Guardian repo. It adds `public/model/paddy/`, `src/lib/diagnosis/__fixtures__/parity-paddy/` and `docs/model-card-paddy.md`.
2. In `src/lib/diagnosis/crops.ts`, change `PHOTO_CROPS` to `['potato', 'paddy']`.
3. `npm test`: the config, labels, advice, questions and parity images are all checked.
4. Try it on real photos from a Bangladeshi field: `npx tsx scripts/eval-field-photos.ts <dir> --crop paddy --write`.